In [1]:
import pandas as pd
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
# -----------------------------
# Base paths
# -----------------------------
BASE_DIR = ".."
data_dir = os.path.join(BASE_DIR, "data")
models_dir = os.path.join(BASE_DIR, "models")
results_dir = os.path.join(BASE_DIR, "results")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# -----------------------------
# Load dataset
# -----------------------------
file_path = os.path.join(data_dir, "standardized_training_data.csv")
df = pd.read_csv(file_path)

# -----------------------------
# Features and labels
# -----------------------------
X = df[['Entropy', 'Energy', 'Correlation', 'Contrast', 'Homogeneity']]
y = df['Encrypt']

# -----------------------------
# Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# (Optional) Save splits
# -----------------------------
X_train.to_csv(os.path.join(data_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(data_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(data_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(data_dir, "y_test.csv"), index=False)

print("Training dataset prepared")

# -----------------------------
# Train SVM model
# -----------------------------
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train, y_train)

# -----------------------------
# Save model
# -----------------------------
model_path = os.path.join(models_dir, "svm_trained_model.pkl")
joblib.dump(svm_model, model_path)

print(f"Model trained and saved at {model_path}")

# -----------------------------
# Predictions
# -----------------------------
y_pred = svm_model.predict(X_test)

# -----------------------------
# Evaluation
# -----------------------------
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", cm)

# -----------------------------
# Save confusion matrix as image
# -----------------------------
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Encrypt', 'Encrypt'],
            yticklabels=['No Encrypt', 'Encrypt'])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("SVM Confusion Matrix")

cm_path = os.path.join(results_dir, "svm_confusion_matrix.png")
plt.savefig(cm_path)
plt.close()

print(f"Confusion matrix image saved at {cm_path}")

# -----------------------------
# Overfitting check
# -----------------------------
train_accuracy = svm_model.score(X_train, y_train) * 100
test_accuracy = svm_model.score(X_test, y_test) * 100

print(f"\nTraining Accuracy: {train_accuracy:.2f}%")
print(f"Testing Accuracy: {test_accuracy:.2f}%")

if train_accuracy - test_accuracy > 5:
    print("Possible Overfitting Detected. Consider regularization or more data.")
else:
    print("No major signs of overfitting.")

Training dataset prepared
Model trained and saved at ..\models\svm_trained_model.pkl

Model Accuracy: 99.35%

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.96      0.98       105
           1       0.99      1.00      1.00       510

    accuracy                           0.99       615
   macro avg       1.00      0.98      0.99       615
weighted avg       0.99      0.99      0.99       615


Confusion Matrix:
 [[101   4]
 [  0 510]]
Confusion matrix image saved at ..\results\svm_confusion_matrix.png

Training Accuracy: 99.76%
Testing Accuracy: 99.35%
No major signs of overfitting.
